## 第17章 使用API

### 17.1 GitHub API

In [ ]:
import requests
import plotly.express as px  # pyright: ignore[reportMissingTypeStubs]

# 调用GitHub API，获取Python仓库列表，注意：有速率限制
url = "https://api.github.com/search/repositories?q=language:python+sort:stars+stars:>10000"
headers = {"Accept": "application/vnd.github.v3+json"}
r = requests.get(url, headers=headers)
print(f"Status code: {r.status_code}")

# 解析基本数据
data = r.json()
print(f"Total repositories: {data['total_count']}")
print(f"Complete results: {not data['incomplete_results']}")

# 解析仓库数据
repos = data["items"]
print(f"Repositories returned: {len(repos)}")
links, stars, infos = [], [], []
for repo in repos:
    # names.append(repo["name"])  # 仓库名称
    links.append(f"<a href='{repo['html_url']}'>{repo['name']}</a>")  # 仓库链接
    stars.append(repo["stargazers_count"])  # 仓库星标数
    infos.append(f"Owner: {repo["owner"]["login"]} <br /> Description: {repo["description"]}")  # 仓库描述

# 绘制仓库星标柱状图
fig = px.bar(x=links, y=stars, hover_name=infos)
fig.update_layout(
    title="Most-Starred Python Projects on GitHub",
    title_x=0.50,
    title_font_size=16,
    xaxis_title="Repository",
    xaxis_title_font_size=12,
    xaxis_tickfont_size=10,
    yaxis_title="Stars",
    yaxis_title_font_size=12,
    yaxis_tickfont_size=10
)
fig.update_traces(marker_color="Red", marker_opacity=0.6)
# fig.write_html("res/github.html")
fig.show()

Status code: 200
Total repositories: 864
Complete results: True
Repositories returned: 30


### 17.2 HackNews API

In [ ]:
import requests
import plotly.express as px  # pyright: ignore[reportMissingTypeStubs]
from operator import itemgetter

# 调用Hacker News API，获取最热文章ID列表
url = "https://hacker-news.firebaseio.com/v0/topstories.json"
r = requests.get(url)
print(f"Status code: {r.status_code}")

# 根据ID列表，获取文章详情
topstories = r.json()
articles = []
for id in topstories[:20]:
    url = f"https://hacker-news.firebaseio.com/v0/item/{id}.json"
    r = requests.get(url)
    resp = r.json()
    try:
        article = {
            "title": resp["title"],
            "link": f"<a href='https://news.ycombinator.com/item?id={id}'>{resp['title'][:20]}</a>",
            "comments": resp["descendants"]
        }
    except KeyError:
        continue
    else:
        articles.append(article)

# 按评论数排序
articles = sorted(articles, key=itemgetter("comments"), reverse=True)
links = [article["link"] for article in articles]
comments = [article["comments"] for article in articles]
titles = [article["title"] for article in articles]

# 绘制文章评论数柱状图
fig = px.bar(x=links, y=comments, hover_name=titles)
fig.update_layout(
    title="Most active discussions on Hacker News",
    title_x=0.5,
    title_font_size=16,
    xaxis_title="Article",
    xaxis_title_font_size=12,
    xaxis_tickfont_size=10,
    yaxis_title="Comments",
    yaxis_title_font_size=12,
    yaxis_tickfont_size=10
)
fig.update_traces(marker_color="SkyBlue", marker_opacity=0.6)
# fig.write_html("res/hackernews.html")
fig.show()

Status code: 200
